<a href="https://colab.research.google.com/github/remideso/remideso.github.io/blob/main/BeltLineCapstone.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# **Tracks of Change: Race, Space, and the Atlanta BeltLine**

This capstone looks at how proximity to the Atlanta BeltLine relates to housing costs and racial composition across Atlanta census tracts.

The main story I am telling is that redevelopment is not just about trails, green space, and investment. It also changes who can afford to stay in a neighborhood. By looking at census tract data from 2000, 2010, and 2020, I use data to show how housing costs and racial composition shifted over time, especially near the BeltLine.

This notebook is my statistical analysis and data preparation section. Before building predictive models, I wanted to understand the structure of the data, test whether the differences were statistically meaningful, and check for issues that could affect future machine learning models.

Unlike a simple one-year analysis, this notebook is organized around time. The goal is to compare 2000, 2010, and 2020 so the results match the larger timeline story of my capstone.


# **Problem Statement**

The Atlanta BeltLine is often presented as a redevelopment success story because it brings trails, parks, public art, and new investment into the city. But redevelopment does not impact everyone equally.

In historically Black neighborhoods, rising rents and home values can create pressure for long-term residents, especially when investment makes the area more attractive to higher-income households. This project asks whether neighborhoods closer to the BeltLine are experiencing patterns that point toward rising housing costs and demographic change over time.

The larger question is not just whether the BeltLine created growth. The question is who benefits from that growth and who may be pushed out by it.


# **Research Question**

How does proximity to the Atlanta BeltLine relate to housing costs and racial composition in Atlanta from 2000 to 2020?

More specifically, I am looking at:

- how income, home value, and rent changed from 2000 to 2010 to 2020
- whether areas closer to the BeltLine changed differently from areas farther away
- whether racial composition differs by distance from the BeltLine in each year
- whether rent, income, home value, race, and distance are statistically related by year
- whether the dataset has issues that need to be handled before machine learning


# **Data Definition**

This dataset includes census tract-level data for Atlanta across three time points: 2000, 2010, and 2020. It combines demographic, housing, economic, and spatial variables.

The final dataset has **854 rows** and **24 original columns**.

Important variables include:

- `year`: census/ACS year
- `median_income`: median household income
- `median_home_value`: median owner-occupied home value
- `median_gross_rent`: median gross rent
- `pct_black`: percent of residents who are Black
- `pct_white`: percent of residents who are White
- `pct_hispanic`: percent of residents who are Hispanic
- `pct_renter`: percent of occupied housing units that are renter occupied
- `poverty_rate`: percent of population below poverty
- `centroid_distance_miles`: tract centroid distance from the BeltLine
- `beltline_half_mile`: tract is within 0.5 miles of the BeltLine
- `beltline_one_mile`: tract is within 1 mile of the BeltLine
- `beltline_three_miles`: tract is within 3 miles of the BeltLine

The year variable is important because this capstone is about change over time. So most of the analysis below separates results for 2000, 2010, and 2020 instead of mixing all years together.


In [ ]:
# import libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# statistical tests
from scipy.stats import ttest_ind, f_oneway, pearsonr

# regression and multicollinearity
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# machine learning prep
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# load the dataset from github

df = pd.read_csv('https://raw.githubusercontent.com/remideso/remideso.github.io/main/atlanta_spatial_final.csv')

# preview the data
df.head()

In [ ]:
# check the size of the dataset

df.shape

In [ ]:
# check columns and data types

df.info()

In [ ]:
# check which years are included

df['year'].value_counts().sort_index()

In [ ]:
# split the data by year

df_2000 = df[df['year'] == 2000].copy()
df_2010 = df[df['year'] == 2010].copy()
df_2020 = df[df['year'] == 2020].copy()

df_2000.shape, df_2010.shape, df_2020.shape

In [ ]:
# create distance groups for comparison

bins = [0, 0.5, 1, 1.5, 3, 5, 10, np.inf]

labels = [
    '0-0.5 miles',
    '0.5-1 mile',
    '1-1.5 miles',
    '1.5-3 miles',
    '3-5 miles',
    '5-10 miles',
    '10+ miles'
]

df['distance_group'] = pd.cut(
    df['centroid_distance_miles'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df_2000 = df[df['year'] == 2000].copy()
df_2010 = df[df['year'] == 2010].copy()
df_2020 = df[df['year'] == 2020].copy()

df[['tract_id', 'year', 'centroid_distance_miles', 'distance_group']].head()

# **1. Exploratory Data Analysis by Year**

The first step is to understand the dataset before trying to predict anything. I started by looking at summary statistics, but I organized them by year.

This matters because my capstone is not just asking what Atlanta looks like overall. It is asking how Atlanta changed from 2000 to 2010 to 2020, especially near the BeltLine.


In [ ]:
# key variables for this analysis

key_vars = [
    'median_income',
    'median_home_value',
    'median_gross_rent',
    'pct_black',
    'pct_white',
    'pct_hispanic',
    'pct_renter',
    'poverty_rate',
    'centroid_distance_miles'
]

# descriptive statistics by year
df.groupby('year')[key_vars].describe()

In [ ]:
# cleaner summary table by year

year_summary = df.groupby('year')[
    ['median_income', 'median_home_value', 'median_gross_rent',
     'pct_black', 'pct_white', 'pct_hispanic', 'pct_renter',
     'poverty_rate']
].mean()

year_summary

## **EDA Interpretation by Year**

Looking across the full dataset by year, the average median income increased from **$41,502** in 2000 to **$54,632** in 2010 and **$78,605** in 2020.

Median home value also increased across the timeline, moving from **$141,607** in 2000 to **$317,217** in 2020. Median gross rent increased from **$632** in 2000 to **$1,321** in 2020.

The racial composition also changed. The average Black population share declined from **65.4%** in 2000 to **46.1%** in 2020. At the same time, the average White population share increased from **26.7%** to **36.0%**.

This is already showing the larger pattern of the project: over time, Atlanta tracts in this dataset became more expensive while the racial makeup also shifted.


# **2. BeltLine Area Change Over Time**

Next, I focused specifically on tracts within 3 miles of the BeltLine. This is important because my capstone is not only about Atlanta overall. It is about areas most connected to BeltLine redevelopment.


In [ ]:
# average values within 3 miles of the BeltLine by year

within3_summary = df[df['beltline_three_miles'] == 1].groupby('year')[
    ['median_income', 'median_home_value', 'median_gross_rent',
     'pct_black', 'pct_white', 'pct_hispanic', 'pct_renter',
     'poverty_rate']
].mean()

within3_summary

In [ ]:
# percent change from 2000 to 2020 within 3 miles

percent_change_2000_2020 = (
    (within3_summary.loc[2020] - within3_summary.loc[2000]) /
    within3_summary.loc[2000]
) * 100

percent_change_2000_2020

## **Within 3 Miles Interpretation**

Within 3 miles of the BeltLine, the changes from 2000 to 2020 are very clear.

Median income increased from **$31,979** in 2000 to **$71,021** in 2020. That is a **122.1% increase**.

Median home value increased from **$127,596** to **$353,061**, which is a **176.7% increase**.

Median gross rent increased from **$555** to **$1,279**, which is a **130.7% increase**.

The racial change is just as important. The average Black population share within 3 miles decreased from **70.9%** in 2000 to **43.9%** in 2020. At the same time, the White population share increased from **22.8%** to **42.2%**.

This is the main BeltLine story in the data: areas near the BeltLine became more expensive, less Black, and more White over time.


In [ ]:
# compare within 3 miles vs outside 3 miles by year

three_mile_comparison = df.groupby(['year', 'beltline_three_miles'])[
    ['median_income', 'median_home_value', 'median_gross_rent',
     'pct_black', 'pct_white', 'poverty_rate']
].mean()

three_mile_comparison

## **Near vs Far Interpretation**

The 3-mile comparison helps separate BeltLine-adjacent areas from areas farther away. This is useful because it shows whether the neighborhoods closer to the BeltLine are moving differently over time.

The important thing here is not just one year by itself. It is the direction of change across 2000, 2010, and 2020. The near-BeltLine tracts show rising income, rising home values, and rising rent over time, while also showing a decline in Black population share.

This supports the idea that redevelopment and demographic change are happening together.


# **3. Visual Distributions by Year**

I used charts to show how the major variables changed over time. Instead of only looking at one overall distribution, these visuals compare 2000, 2010, and 2020.


In [ ]:
# median income over time

plt.figure(figsize=(8,5))
sns.lineplot(data=df, x='year', y='median_income', estimator='mean', marker='o')
plt.title('Median Income Over Time')
plt.xlabel('Year')
plt.ylabel('Average Median Income')
plt.show()

In [ ]:
# median home value over time

plt.figure(figsize=(8,5))
sns.lineplot(data=df, x='year', y='median_home_value', estimator='mean', marker='o')
plt.title('Median Home Value Over Time')
plt.xlabel('Year')
plt.ylabel('Average Median Home Value')
plt.show()

In [ ]:
# median gross rent over time

plt.figure(figsize=(8,5))
sns.lineplot(data=df, x='year', y='median_gross_rent', estimator='mean', marker='o')
plt.title('Median Gross Rent Over Time')
plt.xlabel('Year')
plt.ylabel('Average Median Gross Rent')
plt.show()

In [ ]:
# percent black and percent white over time

race_trend = df.groupby('year')[['pct_black', 'pct_white']].mean().reset_index()

race_long = race_trend.melt(
    id_vars='year',
    value_vars=['pct_black', 'pct_white'],
    var_name='race_group',
    value_name='percent'
)

plt.figure(figsize=(8,5))
sns.lineplot(data=race_long, x='year', y='percent', hue='race_group', marker='o')
plt.title('Average Percent Black and White Over Time')
plt.xlabel('Year')
plt.ylabel('Average Percent')
plt.show()

In [ ]:
# rent distribution by year

plt.figure(figsize=(9,5))
sns.boxplot(data=df, x='year', y='median_gross_rent')
plt.title('Median Gross Rent Distribution by Year')
plt.xlabel('Year')
plt.ylabel('Median Gross Rent')
plt.show()

In [ ]:
# percent black distribution by year

plt.figure(figsize=(9,5))
sns.boxplot(data=df, x='year', y='pct_black')
plt.title('Percent Black Distribution by Year')
plt.xlabel('Year')
plt.ylabel('Percent Black')
plt.show()

## **Visual Interpretation**

The trend charts show a steady increase in income, home values, and rent across the three time points. The boxplots also show that the spread of housing costs changes over time, meaning some tracts became much more expensive than others.

The race trend is especially important for my capstone. The average percent Black declines over time while the average percent White increases. This supports the displacement and neighborhood transformation part of the story.

The visuals make the timeline clearer because they show the change instead of flattening all years into one summary.


# **4. Distance Group Analysis by Year**

Next, I looked at distance groups by year. This helps answer whether the relationship between BeltLine proximity and neighborhood change looks different in 2000, 2010, and 2020.


In [ ]:
# average values by year and distance group

distance_year_summary = df.groupby(['year', 'distance_group'])[
    ['median_income', 'median_home_value', 'median_gross_rent',
     'pct_black', 'pct_white', 'poverty_rate', 'pct_renter']
].mean()

distance_year_summary

In [ ]:
# rent by distance group and year

plt.figure(figsize=(12,6))
sns.lineplot(
    data=df,
    x='distance_group',
    y='median_gross_rent',
    hue='year',
    estimator='mean',
    marker='o'
)
plt.title('Average Rent by BeltLine Distance Group and Year')
plt.xlabel('Distance from BeltLine')
plt.ylabel('Average Median Gross Rent')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# percent black by distance group and year

plt.figure(figsize=(12,6))
sns.lineplot(
    data=df,
    x='distance_group',
    y='pct_black',
    hue='year',
    estimator='mean',
    marker='o'
)
plt.title('Average Percent Black by BeltLine Distance Group and Year')
plt.xlabel('Distance from BeltLine')
plt.ylabel('Average Percent Black')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# home value by distance group and year

plt.figure(figsize=(12,6))
sns.lineplot(
    data=df,
    x='distance_group',
    y='median_home_value',
    hue='year',
    estimator='mean',
    marker='o'
)
plt.title('Average Home Value by BeltLine Distance Group and Year')
plt.xlabel('Distance from BeltLine')
plt.ylabel('Average Median Home Value')
plt.xticks(rotation=45)
plt.show()

## **Distance Group Interpretation by Year**

This section is important because it connects the timeline to space. It shows that the BeltLine relationship is not just about one moment in time. The patterns shift across 2000, 2010, and 2020.

The distance group charts show how rent, home value, and percent Black vary across distance bands. This helps show where redevelopment pressure may be strongest and how that changed over time.

The results also remind me not to oversimplify. Distance matters, but Atlanta neighborhoods have their own histories. So the analysis needs to consider both time and distance together.


# **5. Correlation Analysis by Year**

The correlation matrix helps show how variables move together. Since this project is about change over time, I created separate correlation matrices for 2000, 2010, and 2020.


In [ ]:
# variables for correlation

corr_vars = [
    'median_income',
    'median_home_value',
    'median_gross_rent',
    'pct_black',
    'pct_white',
    'pct_renter',
    'poverty_rate',
    'centroid_distance_miles'
]

# correlation matrix for 2000
corr_2000 = df_2000[corr_vars].corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr_2000, annot=True, cmap='BrBG', center=0, fmt='.2f')
plt.title('Correlation Matrix of Key Variables: 2000')
plt.show()

In [ ]:
# correlation matrix for 2010

corr_2010 = df_2010[corr_vars].corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr_2010, annot=True, cmap='BrBG', center=0, fmt='.2f')
plt.title('Correlation Matrix of Key Variables: 2010')
plt.show()

In [ ]:
# correlation matrix for 2020

corr_2020 = df_2020[corr_vars].corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr_2020, annot=True, cmap='BrBG', center=0, fmt='.2f')
plt.title('Correlation Matrix of Key Variables: 2020')
plt.show()

## **Correlation Interpretation by Year**

The year-specific correlations show how the relationships changed across the timeline.

In 2000, the correlation between median income and rent is **0.638**. In 2010, it is **0.676**, and in 2020 it is **0.719**. This shows that income and rent are related across the full timeline.

For percent Black and median income, the correlation is **-0.746** in 2000, **-0.702** in 2010, and **-0.679** in 2020.

This matters because it shows that race and income are connected in the dataset, but the strength of that relationship can shift over time. That is why it would have been misleading to only use one combined correlation matrix for all years.


# **6. Hypothesis Testing by Year**

For the formal testing section, I ran tests by year instead of combining all years together.

I used:
- T-tests to compare tracts within 1 mile of the BeltLine to tracts farther than 1 mile
- ANOVA to compare multiple distance groups within each year
- Pearson correlation tests by year


## **T-Test: Rent Near the BeltLine vs Farther Away by Year**

**H0:** There is no difference in median gross rent between tracts within 1 mile of the BeltLine and tracts farther than 1 mile for that year.

**H1:** There is a difference in median gross rent between tracts within 1 mile of the BeltLine and tracts farther than 1 mile for that year.


In [ ]:
# t-tests for rent within 1 mile vs farther than 1 mile by year

from scipy.stats import ttest_ind

for yr in [2000, 2010, 2020]:
    temp = df[df['year'] == yr]
    near = temp[temp['centroid_distance_miles'] <= 1]['median_gross_rent'].dropna()
    far = temp[temp['centroid_distance_miles'] > 1]['median_gross_rent'].dropna()

    t_stat, p_value = ttest_ind(near, far, equal_var=False)

    print('\nYear:', yr)
    print('near 1 mile mean rent:', near.mean())
    print('farther than 1 mile mean rent:', far.mean())
    print('t-statistic:', t_stat)
    print('p-value:', p_value)

## **Rent T-Test Interpretation by Year**

In 2000, the average rent within 1 mile of the BeltLine was **$561**, compared to **$661** farther than 1 mile. The p-value was **0.009**.

In 2010, the average rent within 1 mile was **$853**, compared to **$975** farther than 1 mile. The p-value was **0.001**.

In 2020, the average rent within 1 mile was **$1,273**, compared to **$1,331** farther than 1 mile. The p-value was **0.223**.

This is helpful because it shows how the near/far rent gap changes across time. It also shows that a simple near/far test does not tell the whole story by itself, which is why I also use ANOVA across multiple distance groups.


## **T-Test: Home Value Near the BeltLine vs Farther Away by Year**

**H0:** There is no difference in median home value between tracts within 1 mile of the BeltLine and tracts farther than 1 mile for that year.

**H1:** There is a difference in median home value between tracts within 1 mile of the BeltLine and tracts farther than 1 mile for that year.


In [ ]:
# t-tests for home value within 1 mile vs farther than 1 mile by year

for yr in [2000, 2010, 2020]:
    temp = df[df['year'] == yr]
    near = temp[temp['centroid_distance_miles'] <= 1]['median_home_value'].dropna()
    far = temp[temp['centroid_distance_miles'] > 1]['median_home_value'].dropna()

    t_stat, p_value = ttest_ind(near, far, equal_var=False)

    print('\nYear:', yr)
    print('near 1 mile mean home value:', near.mean())
    print('farther than 1 mile mean home value:', far.mean())
    print('t-statistic:', t_stat)
    print('p-value:', p_value)

## **Home Value T-Test Interpretation by Year**

In 2000, the average home value within 1 mile was **$128,205**, compared to **$146,911** farther than 1 mile. The p-value was **0.469**.

In 2010, the average home value within 1 mile was **$200,561**, compared to **$239,165** farther than 1 mile. The p-value was **0.105**.

In 2020, the average home value within 1 mile was **$364,377**, compared to **$308,875** farther than 1 mile. The p-value was **0.037**.

This test is important because home value is one of the clearest indicators of redevelopment pressure. Rising home values can benefit owners, but they can also make neighborhoods less accessible to renters and lower-income residents.


## **T-Test: Percent Black Near the BeltLine vs Farther Away by Year**

**H0:** There is no difference in average percent Black between tracts within 1 mile of the BeltLine and tracts farther than 1 mile for that year.

**H1:** There is a difference in average percent Black between tracts within 1 mile of the BeltLine and tracts farther than 1 mile for that year.


In [ ]:
# t-tests for pct black within 1 mile vs farther than 1 mile by year

for yr in [2000, 2010, 2020]:
    temp = df[df['year'] == yr]
    near = temp[temp['centroid_distance_miles'] <= 1]['pct_black'].dropna()
    far = temp[temp['centroid_distance_miles'] > 1]['pct_black'].dropna()

    t_stat, p_value = ttest_ind(near, far, equal_var=False)

    print('\nYear:', yr)
    print('near 1 mile mean pct black:', near.mean())
    print('farther than 1 mile mean pct black:', far.mean())
    print('t-statistic:', t_stat)
    print('p-value:', p_value)

## **Percent Black T-Test Interpretation by Year**

In 2000, the average percent Black within 1 mile was **72.1%**, compared to **62.8%** farther than 1 mile. The p-value was **0.167**.

In 2010, the average percent Black within 1 mile was **66.1%**, compared to **52.2%** farther than 1 mile. The p-value was **0.036**.

In 2020, the average percent Black within 1 mile was **43.6%**, compared to **46.6%** farther than 1 mile. The p-value was **0.464**.

This test directly connects to the race and displacement part of the capstone. The point is not only whether the BeltLine is associated with higher prices, but whether the racial composition around it changed across time.


# **7. ANOVA by Year**

The t-tests compare only two groups: within 1 mile and farther than 1 mile. But the BeltLine effect may not be that simple. ANOVA lets me compare multiple distance groups within each year.


## **ANOVA: Rent by Distance Group and Year**

**H0:** Average rent is the same across all BeltLine distance groups for that year.

**H1:** At least one distance group has a different average rent for that year.


In [ ]:
# anova for rent across distance groups by year

for yr in [2000, 2010, 2020]:
    temp = df[df['year'] == yr]

    groups = [
        group['median_gross_rent'].dropna().values
        for name, group in temp.groupby('distance_group')
        if len(group['median_gross_rent'].dropna()) > 1
    ]

    f_stat, p_value = f_oneway(*groups)

    print('\nYear:', yr)
    print('F-statistic:', f_stat)
    print('p-value:', p_value)

## **Rent ANOVA Interpretation by Year**

For rent by distance group, the ANOVA p-value was **< 0.001** in 2000, **< 0.001** in 2010, and **0.003** in 2020.

This test is stronger than the simple near/far t-test because it compares more than two distance bands. That matters because redevelopment pressure does not always stop exactly at 1 mile.

The ANOVA results help show whether rent differs across space within each year.


## **ANOVA: Percent Black by Distance Group and Year**

**H0:** Average percent Black is the same across all BeltLine distance groups for that year.

**H1:** At least one distance group has a different average percent Black for that year.


In [ ]:
# anova for percent black across distance groups by year

for yr in [2000, 2010, 2020]:
    temp = df[df['year'] == yr]

    groups = [
        group['pct_black'].dropna().values
        for name, group in temp.groupby('distance_group')
        if len(group['pct_black'].dropna()) > 1
    ]

    f_stat, p_value = f_oneway(*groups)

    print('\nYear:', yr)
    print('F-statistic:', f_stat)
    print('p-value:', p_value)

## **Percent Black ANOVA Interpretation by Year**

For percent Black by distance group, the ANOVA p-value was **0.083** in 2000, **0.170** in 2010, and **0.370** in 2020.

This is important because it shows whether racial composition differs across BeltLine distance groups within each time period. If the p-value is below 0.05, that means at least one distance group has a significantly different average percent Black.

This supports the larger point that race and space are connected in the BeltLine story.


# **8. Pearson Correlation Tests by Year**

Pearson correlation tests whether two numeric variables have a statistically significant linear relationship. I ran these by year to avoid mixing time periods together.


In [ ]:
# pearson correlation tests by year

for yr in [2000, 2010, 2020]:
    temp = df[df['year'] == yr]

    income_rent = temp[['median_income', 'median_gross_rent']].dropna()
    black_income = temp[['pct_black', 'median_income']].dropna()
    distance_home = temp[['centroid_distance_miles', 'median_home_value']].dropna()

    r1, p1 = pearsonr(income_rent['median_income'], income_rent['median_gross_rent'])
    r2, p2 = pearsonr(black_income['pct_black'], black_income['median_income'])
    r3, p3 = pearsonr(distance_home['centroid_distance_miles'], distance_home['median_home_value'])

    print('\nYear:', yr)
    print('income vs rent:', r1, p1)
    print('pct black vs income:', r2, p2)
    print('distance vs home value:', r3, p3)

## **Pearson Correlation Interpretation by Year**

For income and rent, the correlation was **0.638** in 2000, **0.676** in 2010, and **0.719** in 2020.

For percent Black and median income, the correlation was **-0.746** in 2000, **-0.702** in 2010, and **-0.679** in 2020.

For distance and home value, the correlation was **0.186** in 2000, **0.143** in 2010, and **-0.009** in 2020.

Running these by year makes the analysis more accurate because the relationship between housing, race, and distance can change over time.


# **9. OLS Regression by Year**

I used OLS regression to see how much rent can be explained by income, race, poverty, and distance from the BeltLine in each year.

The dependent variable is:

`median_gross_rent`

The independent variables are:
- `median_income`
- `pct_black`
- `poverty_rate`
- `centroid_distance_miles`


In [ ]:
# ols regression by year

for yr in [2000, 2010, 2020]:
    temp = df[df['year'] == yr][
        ['median_gross_rent', 'median_income', 'pct_black',
         'poverty_rate', 'centroid_distance_miles']
    ].dropna()

    X = temp[
        ['median_income', 'pct_black', 'poverty_rate',
         'centroid_distance_miles']
    ]

    y = temp['median_gross_rent']

    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()

    print('\n==============================')
    print('YEAR:', yr)
    print('==============================')
    print(model.summary())

## **OLS Regression Interpretation by Year**

The OLS results show how the relationship between rent and the predictors changes over time.

In 2000, the model had R-squared **0.525**, income coefficient **0.0011**, distance coefficient **9.19**.

In 2010, the model had R-squared **0.521**, income coefficient **0.0036**, distance coefficient **7.94**.

In 2020, the model had R-squared **0.550**, income coefficient **0.0057**, distance coefficient **10.02**.

This is important because rent is not being explained by one factor alone. Income, poverty, race, and distance all interact with Atlanta’s neighborhood history. The year-by-year regression helps avoid flattening the timeline into one overall model.


# **10. Pre-ML Data Diagnosis by Year**

Before machine learning, I need to check for data problems that could affect model performance.

This includes:
- missing values
- outliers
- multicollinearity
- class imbalance

Since this project is about time, I also check these issues by year.


## **Missing Values by Year**

In [ ]:
# missing values by year

missing_by_year = df.groupby('year').apply(lambda x: x.isnull().sum())

missing_by_year

## **Missing Values Interpretation**

Checking missing values by year matters because missingness may not be evenly distributed across 2000, 2010, and 2020.

For this notebook, I did not drop every missing row upfront. Instead, I use `dropna()` inside each statistical test or model so each section only removes the rows needed for that specific analysis.

This keeps more data available and avoids deleting useful rows just because one variable is missing.


## **Outlier Review by Year**

In [ ]:
# outlier boxplots by year

outlier_vars = [
    'median_income',
    'median_home_value',
    'median_gross_rent',
    'pct_black',
    'poverty_rate'
]

for col in outlier_vars:
    plt.figure(figsize=(9,5))
    sns.boxplot(data=df, x='year', y=col)
    plt.title(f'Outlier Check by Year: {col}')
    plt.xlabel('Year')
    plt.ylabel(col)
    plt.show()

## **Outlier Interpretation by Year**

There are outliers in income, home value, rent, and poverty. I kept these outliers because they represent real differences between Atlanta neighborhoods.

For this project, outliers are not just noise. They help show inequality. A very high home value tract or a very high poverty tract is part of the story I am studying.

For machine learning, I may later test transformations or robust models, but I would not delete these values without a strong reason.


## **Multicollinearity by Year**

Multicollinearity happens when independent variables are too strongly related to each other. I checked VIF by year because relationships between features can change over time.


In [ ]:
# vif by year

features = [
    'median_income',
    'median_home_value',
    'median_gross_rent',
    'pct_black',
    'pct_white',
    'pct_renter',
    'poverty_rate',
    'centroid_distance_miles'
]

for yr in [2000, 2010, 2020]:
    temp = df[df['year'] == yr][features].replace([np.inf, -np.inf], np.nan).dropna()

    vif_results = pd.DataFrame()
    vif_results['feature'] = features
    vif_results['VIF'] = [
        variance_inflation_factor(temp.values, i)
        for i in range(temp.shape[1])
    ]

    print('\nYEAR:', yr)
    display(vif_results)

## **Multicollinearity Interpretation by Year**

The VIF results show that income, home value, and rent overlap strongly, especially because they all measure related parts of neighborhood economic conditions.

In 2000, median income had a VIF of **21.75**. In 2010, it was **31.21**. In 2020, it was **25.04**.

This matters because if I use rent as the target variable, I should not also use rent as a predictor. I also need to be careful about including both income and home value in the same model because they may be explaining similar things.

A cleaner feature set for future modeling is:
- median income
- percent Black
- percent renter
- poverty rate
- distance from the BeltLine


In [ ]:
# cleaned vif by year after removing home value and rent from predictors

fixed_features = [
    'median_income',
    'pct_black',
    'pct_renter',
    'poverty_rate',
    'centroid_distance_miles'
]

for yr in [2000, 2010, 2020]:
    temp = df[df['year'] == yr][fixed_features].replace([np.inf, -np.inf], np.nan).dropna()

    fixed_vif_results = pd.DataFrame()
    fixed_vif_results['feature'] = fixed_features
    fixed_vif_results['VIF'] = [
        variance_inflation_factor(temp.values, i)
        for i in range(temp.shape[1])
    ]

    print('\nYEAR:', yr)
    display(fixed_vif_results)

## **Multicollinearity Fix**

After removing home value and rent from the predictor set, the features are cleaner for modeling. This does not mean home value and rent are unimportant. It means I need to be careful about using variables that are measuring the same underlying pattern.

For the machine learning section, I would use a cleaner set of features and avoid having the target variable also appear as a predictor.


## **Class Imbalance by Year**

For classification, I need to check whether the target variable is balanced. Here, I use `beltline_three_miles` as a possible classification target.


In [ ]:
# class imbalance by year

df.groupby('year')['beltline_three_miles'].value_counts()

In [ ]:
# class percentages by year

df.groupby('year')['beltline_three_miles'].value_counts(normalize=True) * 100

## **Class Imbalance Interpretation by Year**

The class balance is fairly consistent by year because each year has the same tract structure repeated across time.

For 2000, about **53.0%** of observations are within 3 miles of the BeltLine. For 2010, about **38.0%** are within 3 miles. For 2020, about **29.4%** are within 3 miles.

This is somewhat imbalanced, but not extreme. If I use this as a classification target, I would use a stratified train-test split and possibly class weights so the model does not mostly learn the larger class.


In [ ]:
# example stratified train-test split for future classification

features_for_ml = [
    'median_income',
    'pct_black',
    'pct_renter',
    'poverty_rate',
    'centroid_distance_miles',
    'year'
]

target = 'beltline_three_miles'

ml_df = df[features_for_ml + [target]].dropna()

X = ml_df[features_for_ml]
y = ml_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('training target balance:')
print(y_train.value_counts(normalize=True) * 100)

print('\ntesting target balance:')
print(y_test.value_counts(normalize=True) * 100)

In [ ]:
# example class weights if needed

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))

class_weight_dict

## **Class Imbalance Fix**

To handle class imbalance, I would use a stratified train-test split. This keeps the class proportions similar in both the training and testing data.

If the smaller class still performs poorly, I would try class weights. I would use class weights before SMOTE because this dataset is spatial and census-based, so synthetic observations could distort the geography.


# **11. Final Model-Ready Dataset**

This section creates a cleaned version of the dataset that can be used for the machine learning section of my capstone.

Since the project is about change over time, I keep `year` as a feature in the model-ready dataset.


In [ ]:
# create model-ready dataset

model_features = [
    'median_income',
    'pct_black',
    'pct_renter',
    'poverty_rate',
    'centroid_distance_miles',
    'year'
]

model_target = 'beltline_three_miles'

model_ready = df[model_features + [model_target]].dropna()

model_ready.head()

In [ ]:
# check final model-ready shape

model_ready.shape

In [ ]:
# scale numeric features for models that need scaling

scaler = StandardScaler()

scaled_features = scaler.fit_transform(model_ready[model_features])

scaled_df = pd.DataFrame(
    scaled_features,
    columns=model_features
)

scaled_df[model_target] = model_ready[model_target].values

scaled_df.head()

# **Final Summary**

This analysis shows that the BeltLine story is not just about redevelopment. It is also about cost, race, space, and time.

The strongest findings are:

- Across the full dataset, median income increased from **$41,502** in 2000 to **$78,605** in 2020.
- Across the full dataset, median rent increased from **$632** to **$1,321**.
- Within 3 miles of the BeltLine, median income increased from **$31,979** to **$71,021**.
- Within 3 miles of the BeltLine, median home value increased by **176.7%** from 2000 to 2020.
- Within 3 miles of the BeltLine, median gross rent increased by **130.7%** from 2000 to 2020.
- Within 3 miles, the average Black population share declined from **70.9%** to **43.9%**.
- Within 3 miles, the average White population share increased from **22.8%** to **42.2%**.
- The t-tests and ANOVA tests were run by year so the analysis matches the actual timeline of the capstone.
- The correlation and OLS regression sections were also separated by year, which avoids mixing 2000, 2010, and 2020 into one combined pattern.
- Multicollinearity was present, especially between income, home value, and rent, so the feature set needs to be cleaned before machine learning.
- The possible classification target is somewhat imbalanced, so stratified splitting and class weights should be considered.

Overall, the data supports my capstone argument: redevelopment around the BeltLine is connected to rising housing costs and demographic change over time. The pattern is not simple, but it is clear enough to show that proximity, race, housing costs, and time should be studied together.


# **Portfolio / GitHub Notes**

For my portfolio, this notebook should be uploaded directly into the `remideso.github.io` repository.

This notebook pulls the CSV directly from GitHub using:

```python
df = pd.read_csv('https://raw.githubusercontent.com/remideso/remideso.github.io/main/atlanta_spatial_final.csv')
```

Suggested file name:

```text
BeltLineCapstone.ipynb
```

Suggested README link:

```markdown
[View BeltLine Capstone Statistical Analysis](BeltLineCapstone.ipynb)
```

To show outputs on GitHub:
1. Open the notebook in Google Colab.
2. Run all cells.
3. Save a copy back to GitHub.
4. Open the notebook in GitHub and confirm charts/tables are visible.
